In [ ]:
# 导入必要的库
import h5py
import numpy as np
import os
import glob
import pandas as pd
from tqdm import tqdm

# 设置基础路径
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data'  # 可根据实际情况修改
data_path = os.path.join(base_dir, 'DATA/TRAIN38.mat')
output_base_dir = os.path.join(base_dir, 'reorganized_data')  # 重建数据的输出目录

# 创建输出目录
os.makedirs(output_base_dir, exist_ok=True)

print(f"数据路径: {data_path}")
print(f"输出目录: {output_base_dir}")

# 1. 数据加载阶段
print("正在加载数据文件...")
f = h5py.File(data_path, 'r')
arrays = {}
for k, v in f.items():
    print(f"加载键: {k}, 形状: {v.shape}")
    arrays[k] = np.array(v)
f.close()

# 提取数据并转置
data = arrays['data'].transpose()  # 体素特征数据
region = arrays['region'].transpose()  # 102维的区域标签
prob_idx = arrays['prob_idx'].transpose()  # 病人ID

# 处理age字段（如果存在）
if 'age' in arrays:
    age_raw = arrays['age']
    # 根据age的维度决定是否需要转置
    if len(age_raw.shape) > 1 and age_raw.shape[0] > 1:
        age = age_raw.transpose()
        print("age数据已转置")
    else:
        age = age_raw
        print("age数据未转置，保持原始形状")
    
    print(f"age数据形状: {age.shape}")
    print(f"数据中的一些age值: {age[:10]}")
else:
    age = None
    print("未找到age数据")

# 2. 数据分析阶段
print("\n正在分析数据集基本信息...")
print(f"数据形状: {data.shape}")
print(f"标签形状: {region.shape}")
print(f"病人索引形状: {prob_idx.shape}")
if age is not None:
    print(f"年龄数据形状: {age.shape}")

# 分析唯一的病人ID
unique_prob_idx = np.unique(prob_idx)
print(f"唯一病人ID: {unique_prob_idx}")
print(f"病人总数: {len(unique_prob_idx)}个")

# 分析区域标签分布
if region.ndim == 2:
    # 计算每个区域标签中有多少个体素
    label_counts = np.sum(region, axis=0)
    active_regions = []
    for i in range(region.shape[1]):
        if label_counts[i] > 0:
            active_regions.append(i)
            print(f"区域 {i}: {label_counts[i]} 个体素")
    
    print(f"活跃区域数量: {len(active_regions)}")
    
    # 计算每个体素被分配到了几个区域
    region_per_voxel = np.sum(region, axis=1)
    unique_counts, count_freqs = np.unique(region_per_voxel, return_counts=True)
    for count, freq in zip(unique_counts, count_freqs):
        print(f"{count} 个区域标签的体素数量: {freq}")
else:
    print("标签不是二维的，无法分析区域分布")

# 3. 数据组织和保存阶段
print("\n正在按病人ID组织和保存数据...")

# 创建总索引文件
dataset_index_file = os.path.join(output_base_dir, "dataset_index.csv")
dataset_index_data = []

# 为每个病人处理数据
for patient_id in tqdm(unique_prob_idx, desc="处理病人数据"):
    patient_id = int(patient_id)
    # 创建病人文件夹
    patient_dir = os.path.join(output_base_dir, f"patient_{patient_id}")
    os.makedirs(patient_dir, exist_ok=True)
    
    # 提取该病人的数据
    patient_indices = np.where(prob_idx == patient_id)[0]
    patient_data = data[patient_indices]
    patient_region = region[patient_indices]
    patient_age = age[patient_indices] if age is not None else None
    
    print(f"\n病人 {patient_id}: {len(patient_indices)} 个样本")
    
    # 创建病人区域索引文件
    patient_index_file = os.path.join(patient_dir, "region_index.csv")
    patient_index_data = []
    
    # 处理102个区域
    for region_id in range(patient_region.shape[1]):
        # 找出该区域的所有体素
        region_indices = np.where(patient_region[:, region_id] == 1)[0]
        
        if len(region_indices) == 0:
            # 该区域没有体素，跳过
            continue
        
        # 提取该区域的体素数据
        region_data = patient_data[region_indices]
        
        # 体素数量
        voxel_count = len(region_indices)
        
        # 保存文件名
        file_name = f"patient_{patient_id}_region_{region_id}_voxels_{voxel_count}.npy"
        file_path = os.path.join(patient_dir, file_name)
        
        # 保存为npy文件
        np.save(file_path, region_data)
        
        # 添加到病人索引数据
        patient_index_data.append({
            "region_id": region_id,
            "voxel_count": voxel_count,
            "file_name": file_name
        })
        
        # 添加到总索引数据
        dataset_index_data.append({
            "patient_id": patient_id,
            "region_id": region_id,
            "voxel_count": voxel_count,
            "file_path": os.path.join(f"patient_{patient_id}", file_name)
        })
        
        print(f"  保存区域 {region_id}: {voxel_count} 个体素")
    
    # 保存病人区域索引文件
    if patient_index_data:
        pd.DataFrame(patient_index_data).to_csv(patient_index_file, index=False)
        print(f"  保存病人索引文件: {patient_index_file}")

# 保存总索引文件
if dataset_index_data:
    pd.DataFrame(dataset_index_data).to_csv(dataset_index_file, index=False)
    print(f"\n保存总索引文件: {dataset_index_file}")



# 4. 验证处理结果
print("\n验证处理结果...")

# 检查输出目录结构
patient_dirs = [d for d in os.listdir(output_base_dir) if os.path.isdir(os.path.join(output_base_dir, d)) and d.startswith("patient_")]
print(f"创建的病人目录数: {len(patient_dirs)}")

# 检查总索引文件
if os.path.exists(dataset_index_file):
    dataset_index = pd.read_csv(dataset_index_file)
    print(f"总索引文件条目数: {len(dataset_index)}")
    
    # 统计每个病人的区域数量
    patient_stats = dataset_index.groupby('patient_id')['region_id'].count()
    print("\n每个病人的区域数量:")
    print(patient_stats)
    
    # 统计每个区域的总体素数量
    region_stats = dataset_index.groupby('region_id')['voxel_count'].sum()
    print("\n每个区域的总体素数量:")
    print(region_stats)

print("\n数据集重建完成！数据已按病人ID和区域标签分别处理并保存，没有进行标准化处理。")